# Lost-words / Zgubione słowa — AI Video + Lip-Sync Pipeline

Polski pipeline do pionowych shortów w stylu filmowych scenariuszy „co by było, gdyby…”, z konkretnymi działaniami bohaterów, narastającą stawką i filmowym montażem.

Przepływ: polski scenariusz dialogowy → osobny polski głos dla każdej kwestii → FLUX still → Kling Image-to-Video → Sync Lipsync dla widocznych mówiących postaci → FFmpeg → MP4 1080×1920.

Wszystkie tytuły, narracje, dialogi i napisy są po polsku. Prompty wizualne są po angielsku. Sceny mają jawny typ: talking_character (widoczna mówiąca postać, wyraźny ruch ciała i lip-sync) albo environment (ruch otoczenia, głos spoza kadru/radio). W każdej scenie talking_character pole motion_prompt opisuje widoczną akcję bohatera — sam ruch kamery nie wystarcza.

## Krótkie ujęcia zamiast jednego długiego kadru

Kling V3 otrzymuje listę `multi_prompt`: jedna wypowiedź jest dzielona na kilka krótkich ujęć o docelowej długości około 2 sekund. Przykładowo 7-sekundowa kwestia dostaje zwykle 4 cięcia: 2 + 2 + 2 + 1 sekunda. Zmieniają się plany i detale, a głos i napisy pozostają ciągłe.

## Ważne: tryb testowy i koszty

Domyślnie TEST_MODE = True. W tym trybie notebook przygotowuje scenariusz i osobne pliki mowy, zapisuje plan generowania i **pomija płatne wywołania FLUX, Kling, Lip-sync oraz eksport MP4**. Nie tworzy prostokątnych plansz ani filmu udającego pełny materiał.

Aby uruchomić pełne generowanie:
1. Dodaj FAL_KEY w Colab Secrets.
2. Zmień TEST_MODE = False w komórce 2.
3. Opcjonalnie dodaj OPENAI_API_KEY, aby generować nowy scenariusz na podstawie własnego tematu; bez niego używany jest polski scenariusz demonstracyjny.
4. Uruchom wszystkie komórki.

Generowanie obrazów, klipów i lip-sync w fal.ai jest rozliczane według aktualnego cennika danego modelu. Jeden odcinek może uruchomić wiele zadań: obraz + Image-to-Video dla każdej kwestii oraz dodatkowy lip-sync dla każdej sceny talking_character. Najpierw sprawdź koszt w panelu fal.ai. Domyślny TEST_MODE chroni przed przypadkowym uruchomieniem zadań.

## Używane modele fal.ai

- FLUX.1 Dev: tekst → realistyczny kadr startowy.
- Kling Video 3 Standard Image-to-Video: kadr → animowany klip.
- Sync Lipsync 2: klip twarzy + osobne audio → ruch ust zsynchronizowany z wypowiedzią.

Dokumentacja: [FLUX.1 Dev](https://fal.ai/models/fal-ai/flux/dev/api), [Kling Video 3 Standard I2V](https://fal.ai/models/fal-ai/kling-video/v3/standard/image-to-video/api), [Sync Lipsync 2](https://fal.ai/models/fal-ai/sync-lipsync/v2/api), [fal Queue API](https://fal.ai/docs/api-reference/inference-api/queue/submit-path).

## Colab Secrets

W zakładce Secrets dodaj:

| Secret | Kiedy potrzebny |
|---|---|
| FAL_KEY | Wymagany dla pełnego generowania obrazów, I2V i lip-sync przez fal.ai |
| OPENAI_API_KEY | Opcjonalny: dynamiczny scenariusz po polsku; można użyć lokalnego scenariusza demo bez tego klucza |
| OPENAI_BASE_URL | Opcjonalny: domyślnie https://api.openai.com/v1 |
| OPENAI_CHAT_MODEL | Opcjonalny: domyślnie gpt-4o-mini |
| FAL_WEBHOOK_URL | Opcjonalny: publiczny HTTPS endpoint do powiadomień fal; Colab i tak monitoruje status przez polling |
| TTS_API_KEY | Opcjonalny: miejsce na własny adapter TTS; domyślny Edge TTS nie potrzebuje klucza |

Nie wpisuj kluczy do komórek ani repozytorium. Colab Secrets udostępniaj wyłącznie temu notebookowi. Materiały przesłane do fal są przechowywane na ich CDN zgodnie z ustawieniami retencji konta i usługi; nie przesyłaj prywatnych nagrań ani wizerunków bez odpowiedniej zgody.

## Jak działa notebook

1. Montuje Google Drive w /content/drive/MyDrive/lost_words_project i instaluje FFmpeg, DejaVu Sans, requests, Pillow, edge-tts i fal-client.
2. Konfiguruje temat, lokację, 35–45 sekund, liczbę kwestii oraz TEST_MODE.
3. Tworzy lub waliduje polski JSON: tytuł, hook i sceny z polami scene_id, speaker, voice_role, text, visual_prompt, shot_type, character_id i character_description.
4. Generuje osobny plik audio dla każdej kwestii, z różnymi głosami PL, tempem, pitch i filtracją radiową.
5. W trybie pełnym tworzy realistyczny obraz startowy FLUX dla każdej sceny.
6. Wysyła obraz do kolejki Kling I2V; dla talking_character przekazuje klip i audio do Sync Lipsync 2.
7. Pobiera klipy, synchronizuje z głosami, dokleja pauzy, ambient i polskie napisy, eksportuje H.264/AAC.
8. ffprobe sprawdza rozmiar, rozdzielczość, strumień wideo, strumień audio i długość; dodatkowo FFmpeg sprawdza, czy audio nie jest cyfrową ciszą.

## Gdzie trafiają pliki

- MP4: MyDrive/lost_words_project/output/
- Scenariusze i manifesty: MyDrive/lost_words_project/assets/
- Audio kwestii: MyDrive/lost_words_project/audio/
- Obrazy startowe: MyDrive/lost_words_project/frames/
- Animowane klipy i materiały po lip-sync: MyDrive/lost_words_project/video/
- Pliki tymczasowe: MyDrive/lost_words_project/temp/

## Debugowanie

Wklej numer komórki, pełny traceback od Traceback (most recent call last): do końca oraz poprzedzające komunikaty [INFO], [QUEUE], [OSTRZEŻENIE] i [BŁĄD]. Usuń wartości kluczy API, nagłówki Authorization oraz prywatne URL-e. Zastąp je [UKRYTO].

Notebook nie uruchamia płatnych zadań, dopóki TEST_MODE pozostaje True. Jeśli fal zwróci błąd lub limit, zadanie kończy się czytelnym komunikatem zamiast cichego eksportu niekompletnego wideo.


In [ ]:
# KOMÓRKA 1/7 — Google Drive, FFmpeg, biblioteki i wspólne funkcje API
from pathlib import Path
import os, sys, json, re, math, time, random, shutil, subprocess, textwrap, importlib.util
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    raise RuntimeError(
        "Nie udało się zamontować Google Drive. Uruchom notebook w Colab i zaakceptuj dostęp do Dysku. "
        + str(exc)
    ) from exc

PROJECT_ROOT = Path('/content/drive/MyDrive/lost_words_project')
ASSETS_DIR = PROJECT_ROOT / 'assets'
AUDIO_DIR = PROJECT_ROOT / 'audio'
FRAMES_DIR = PROJECT_ROOT / 'frames'
VIDEO_DIR = PROJECT_ROOT / 'video'
TEMP_DIR = PROJECT_ROOT / 'temp'
OUTPUT_DIR = PROJECT_ROOT / 'output'
CHECKPOINTS_ROOT = PROJECT_ROOT / 'checkpoints'
VISUALS_DIR = PROJECT_ROOT / 'visuals'
MODELS_DIR = PROJECT_ROOT / 'models'
for folder in (PROJECT_ROOT, ASSETS_DIR, AUDIO_DIR, FRAMES_DIR, VIDEO_DIR, TEMP_DIR, OUTPUT_DIR, CHECKPOINTS_ROOT, VISUALS_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def run_cmd(args, *, check=True, capture_output=True):
    """Uruchamia proces z diagnostyką stderr."""
    try:
        return subprocess.run(
            [str(x) for x in args],
            check=check,
            text=True,
            stdout=subprocess.PIPE if capture_output else None,
            stderr=subprocess.PIPE if capture_output else None,
        )
    except subprocess.CalledProcessError as exc:
        print(f'[BŁĄD] Proces {args[0]} zakończył się kodem {exc.returncode}.')
        if exc.stderr:
            print(exc.stderr[-5000:])
        raise
    except FileNotFoundError as exc:
        raise RuntimeError(f'Nie znaleziono programu {args[0]}: {exc}') from exc


try:
    if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
        run_cmd(['apt-get', 'update', '-qq'], capture_output=False)
        run_cmd([
            'apt-get', 'install', '-y', '-qq',
            'ffmpeg', 'fonts-dejavu-core', 'fontconfig'
        ], capture_output=False)
    FONT_FILE = Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
    if not FONT_FILE.exists():
        run_cmd(['apt-get', 'install', '-y', '-qq', 'fonts-dejavu-core'], capture_output=False)
    if not FONT_FILE.exists():
        raise RuntimeError(f'Brak czcionki DejaVu Sans: {FONT_FILE}')
except Exception as exc:
    raise RuntimeError(f'Instalacja FFmpeg/fontów nie powiodła się: {exc}') from exc

try:
    run_cmd([
        sys.executable, '-m', 'pip', 'install', '-q',
        'requests>=2.31.0', 'Pillow>=10.0.0', 'edge-tts>=6.1.0'
    ])
    import requests
    from PIL import Image
except Exception as exc:
    raise RuntimeError(f'Instalacja/import pakietów nie powiodły się: {exc}') from exc

print('[OK] FFmpeg:', run_cmd(['ffmpeg', '-version']).stdout.splitlines()[0])
print('[OK] DejaVu Sans:', FONT_FILE)
print('[OK] Projekt:', PROJECT_ROOT)


def probe_duration(path):
    result = run_cmd([
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        str(path),
    ])
    duration = float(result.stdout.strip())
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError(f'Niepoprawna długość pliku {path}: {duration}')
    return duration



def write_json_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + '.tmp')
    temp_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temp_path, path)
    return path


def read_json_if_exists(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError) as exc:
        print(f'[OSTRZEŻENIE] Nie można odczytać checkpointu {path.name}: {exc}')
        return default


def probe_media(path):
    result = run_cmd([
        'ffprobe', '-v', 'error', '-show_streams', '-show_format',
        '-of', 'json', str(path)
    ])
    data = json.loads(result.stdout)
    video = next((x for x in data.get('streams', []) if x.get('codec_type') == 'video'), None)
    audio = next((x for x in data.get('streams', []) if x.get('codec_type') == 'audio'), None)
    return {'data': data, 'video': video, 'audio': audio}


In [ ]:
# KOMÓRKA 2/7 — konfiguracja, TEST_MODE i sekrety

def get_secret(name, default=None):
    """Pobieranie Colab Secrets z awaryjnym odczytem z os.environ."""
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        value = os.environ.get(name)
    return value if value else default


SERIES_NAME = 'Lost-words / Zgubione słowa'
EPISODE_TOPIC = 'Masz osiem godzin, by dotrzeć do punktu ewakuacji. Na aktualnych mapach tego miejsca nie ma od dwudziestu lat.'
EPISODE_LOCATION = 'Nieczynna stacja kolejowa i puste przedmieścia Łodzi'
NUM_SCENES = 14
MIN_VIDEO_SECONDS = 40
GOAL_VIDEO_SECONDS = 60
MAX_VIDEO_SECONDS = 120
TARGET_SECONDS = 75
MIN_SCRIPT_WORDS = 150
VIDEO_WIDTH, VIDEO_HEIGHT, FPS = 1080, 1920, 30
STORY_STYLE_PROFILE = (
    'Immersive what-if survival scenario: a clear premise, a ticking deadline, limited resources, '
    'cause-and-effect escalation, one unsettling contradiction, and a consequential final choice or twist. '
    'Address the viewer in second person when natural. Keep the scenario grounded and cinematic.'
)
VISUAL_STYLE = (
    'cinematic grounded survival thriller, near-future speculative realism, documentary camera language, '
    'desaturated natural colors, subtle 35mm film grain, practical lighting, atmospheric locations, '
    'realistic human performances, expressive close-ups and purposeful environmental shots'
)

# TEST_MODE=True przygotowuje scenariusz/TTS/manifest, ale pomija model wizualny.
# TEST_MODE=False uruchamia lokalny renderer Claude; notebook nie wysyła płatnych zadań API.
TEST_MODE = True
DRY_RUN = TEST_MODE
RESUME_RENDERING = True
LLM_PROVIDER = 'fallback'            # można zmienić na openai_compatible
TTS_PROVIDER = 'edge'
PAUSE_BETWEEN_LINES = 0.0
AMBIENT_MIX_GAIN = 0.12
RADIO_STATIC_GAIN = 0.08
SHOT_DURATION_MIN = 1.0
SHOT_DURATION_TARGET = 2.0
SHOT_DURATION_MAX = 3.0
VISUAL_RENDERER_PATH = VISUALS_DIR / 'shot_renderer.py'
VISUAL_RENDERER_SOURCE_URL = 'https://raw.githubusercontent.com/simonexex544/lost-words-video-pipeline/main/visuals/shot_renderer.py'
EPISODE_ID_OVERRIDE = ''  # wpisz poprzedni EPISODE_ID, aby wznowić render

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
OPENAI_BASE_URL = (get_secret('OPENAI_BASE_URL', 'https://api.openai.com/v1') or '').rstrip('/')
OPENAI_CHAT_MODEL = get_secret('OPENAI_CHAT_MODEL', 'gpt-4o-mini')
if not 10 <= NUM_SCENES <= 18:
    raise ValueError('NUM_SCENES ustaw w zakresie 10–18.')
if not MIN_VIDEO_SECONDS <= TARGET_SECONDS <= MAX_VIDEO_SECONDS:
    raise ValueError('TARGET_SECONDS musi mieścić się w zakresie 40–120 sekund.')
if not 0.05 <= AMBIENT_MIX_GAIN <= 0.30:
    raise ValueError('AMBIENT_MIX_GAIN ustaw w zakresie 0.05–0.30.')
if not SHOT_DURATION_MIN <= SHOT_DURATION_TARGET <= SHOT_DURATION_MAX:
    raise ValueError('Docelowy czas ujęcia musi mieścić się w zakresie 1–3 s.')
if SHOT_DURATION_TARGET not in {1, 2, 3}:
    raise ValueError('SHOT_DURATION_TARGET musi wynosić 1, 2 albo 3 sekundy.')
if not 2 <= MAX_SHOTS_PER_SEGMENT <= 6:
    raise ValueError('MAX_SHOTS_PER_SEGMENT musi mieścić się w zakresie 2–6.')

EPISODE_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
EPISODE_ID = (EPISODE_ID_OVERRIDE.strip() if EPISODE_ID_OVERRIDE else re.sub(r'[^a-z0-9]+', '-', f'lost-words-{EPISODE_STAMP}'.lower()).strip('-'))
FRAME_EPISODE_DIR = FRAMES_DIR / EPISODE_ID
SHOT_FRAME_DIR = FRAME_EPISODE_DIR / 'shots'
VIDEO_EPISODE_DIR = VIDEO_DIR / EPISODE_ID
SHOT_VIDEO_DIR = VISUALS_DIR / EPISODE_ID / 'shots'
CHECKPOINT_DIR = CHECKPOINTS_ROOT / EPISODE_ID
AUDIO_EPISODE_DIR = AUDIO_DIR / EPISODE_ID
STORY_PATH = ASSETS_DIR / f'{EPISODE_ID}_story.json'
SHOT_PLAN_PATH = ASSETS_DIR / f'{EPISODE_ID}_shot_manifest.json'
VISUAL_MANIFEST_PATH = ASSETS_DIR / f'{EPISODE_ID}_visual_manifest.json'
for folder in (FRAME_EPISODE_DIR, SHOT_FRAME_DIR, VIDEO_EPISODE_DIR, SHOT_VIDEO_DIR, CHECKPOINT_DIR, AUDIO_EPISODE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

CHARACTER_BIBLE = {
    'survivor_1': (
        'fictional adult Polish male survivor, around 35 years old, tired hazel eyes, '
        'short dark hair, worn charcoal railway jacket, small old scar above eyebrow, '
        'consistent face and clothes in every shot'
    ),
    'survivor_2': (
        'fictional adult Polish female survivor, around 30 years old, pale tired face, '
        'dark hair tied back, faded olive coat, small analogue radio headset, '
        'consistent face and clothes in every shot'
    ),
    'unknown_entity': (
        'fictional adult unknown passenger, middle-aged, very pale face, wet dark coat, '
        'motionless unsettling expression, consistent appearance in every shot'
    ),
}

print(f'[OK] Seria: {SERIES_NAME} | temat: {EPISODE_TOPIC}')
print(f'[OK] Lokacja: {EPISODE_LOCATION} | sceny: {NUM_SCENES} | cel: {TARGET_SECONDS}s')
print(f'[INFO] Długość akceptowana: {MIN_VIDEO_SECONDS}–{MAX_VIDEO_SECONDS}s, cel minimum {GOAL_VIDEO_SECONDS}s')
print('[INFO] TEST_MODE:', TEST_MODE, '| lokalny renderer:', 'POMINIĘTY' if TEST_MODE else 'WŁĄCZONY')
print('[INFO] EPISODE_ID:', EPISODE_ID)
print('[INFO] Renderer:', VISUAL_RENDERER_PATH)
print('[INFO] OPENAI_API_KEY:', 'dostępny' if OPENAI_API_KEY else 'brak — scenariusz demo')


In [ ]:
# KOMÓRKA 3/7 — polski scenariusz dialogowy i walidacja JSON

ALLOWED_SPEAKERS = {'narrator', 'ocalaly_1', 'ocalaly_2', 'glos_z_radia', 'nieznany'}
ALLOWED_VOICE_ROLES = {'calm_dark', 'tense_male', 'calm_female', 'radio_distorted', 'unknown'}
ALLOWED_SHOT_TYPES = {'talking_character', 'environment'}


def strip_json_fence(raw_text):
    if not isinstance(raw_text, str) or not raw_text.strip():
        raise ValueError('Model zwrócił pustą odpowiedź.')
    fence = chr(96) * 3
    cleaned = re.sub(r'^\s*' + re.escape(fence) + r'(?:json)?\s*', '', raw_text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r'\s*' + re.escape(fence) + r'\s*$', '', cleaned)
    start, end = cleaned.find('{'), cleaned.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('Nie znaleziono kompletnego obiektu JSON.')
    return cleaned[start:end + 1]


def normalize_scene(item, label):
    if not isinstance(item, dict):
        raise TypeError(f'{label} musi być obiektem JSON.')
    required = (
        'scene_id', 'speaker', 'voice_role', 'text', 'visual_prompt', 'motion_prompt',
        'shot_type', 'character_id', 'character_description'
    )
    missing = [
        field for field in required
        if field not in item or not isinstance(item[field], str)
    ]
    if missing:
        raise ValueError(f'{label}: brak wymaganych pól: {", ".join(missing)}')

    scene_id = item['scene_id'].strip()
    if not re.fullmatch(r'[A-Za-z0-9_-]{1,20}', scene_id):
        raise ValueError(f'{label}: nieprawidłowe scene_id.')
    if item['speaker'] not in ALLOWED_SPEAKERS:
        raise ValueError(f'{label}: nieznany speaker={item["speaker"]!r}.')
    if item['voice_role'] not in ALLOWED_VOICE_ROLES:
        raise ValueError(f'{label}: nieznany voice_role={item["voice_role"]!r}.')
    if item['shot_type'] not in ALLOWED_SHOT_TYPES:
        raise ValueError(f'{label}: shot_type musi być talking_character albo environment.')

    spoken_text = re.sub(r'\s+', ' ', item['text']).strip()
    if not spoken_text:
        raise ValueError(f'{label}: text nie może być pusty.')
    if len(spoken_text.split()) > 24:
        raise ValueError(f'{label}: kwestia jest za długa — podziel ją na krótsze zdania.')
    if not re.search(r'[.!?…]$', spoken_text):
        spoken_text += '.'

    character_id = item['character_id'].strip() or 'none'
    character_description = item['character_description'].strip()
    if item['shot_type'] == 'talking_character':
        if item['speaker'] in {'narrator', 'glos_z_radia'}:
            raise ValueError(f'{label}: narrator i głos z radia powinny być ujęciem environment.')
        if not character_description:
            character_description = CHARACTER_BIBLE.get(character_id, '')
        if not character_description:
            raise ValueError(f'{label}: talking_character wymaga character_description.')
    if item['shot_type'] == 'environment' and item['speaker'] in {'narrator', 'glos_z_radia'}:
        character_id = 'none'

    visual_prompt = item['visual_prompt'].strip()
    motion_prompt = item['motion_prompt'].strip()
    if not motion_prompt:
        raise ValueError(f'{label}: motion_prompt nie może być pusty.')
    if item['shot_type'] == 'talking_character' and len(motion_prompt) < 25:
        raise ValueError(f'{label}: motion_prompt dla postaci musi opisywać konkretną akcję.')
    if item['shot_type'] == 'talking_character':
        visual_prompt += (
            ' Single fictional adult character, stable medium close-up, face and mouth clearly visible, '
            'front-facing or gentle three-quarter angle, lips relaxed, no cuts, no text in image.'
        )
    else:
        visual_prompt += ' Environmental survival-horror shot, no visible speaking face, no text in image.'

    return {
        'scene_id': scene_id,
        'speaker': item['speaker'],
        'voice_role': item['voice_role'],
        'text': spoken_text,
        'visual_prompt': visual_prompt,
        'motion_prompt': motion_prompt,
        'shot_type': item['shot_type'],
        'character_id': character_id,
        'character_description': character_description,
    }


def validate_story(payload):
    if not isinstance(payload, dict):
        raise TypeError('Scenariusz musi być obiektem JSON.')
    if not isinstance(payload.get('title'), str) or not payload['title'].strip():
        raise ValueError('Brak polskiego tytułu.')
    hook = normalize_scene(payload.get('hook'), 'hook')
    if len(hook['text'].split()) > 9:
        raise ValueError('Hook musi mieć maksymalnie 9 słów, aby zmieścił się w pierwszych 3 sekundach.')
    scenes = payload.get('scenes')
    if not isinstance(scenes, list) or len(scenes) != NUM_SCENES:
        raise ValueError(f'Wymagane jest dokładnie {NUM_SCENES} scen.')
    normalized, ids = [], {hook['scene_id']}
    for index, scene in enumerate(scenes, start=1):
        clean = normalize_scene(scene, f'scenes[{index-1}]')
        if clean['scene_id'] in ids:
            raise ValueError(f'Powtórzone scene_id: {clean["scene_id"]}')
        ids.add(clean['scene_id'])
        normalized.append(clean)
    talking_speakers = {
        item['speaker'] for item in [hook] + normalized
        if item['shot_type'] == 'talking_character'
    }
    if not talking_speakers:
        raise ValueError('Scenariusz musi zawierać co najmniej jedną widoczną mówiącą postać.')
    return {'title': payload['title'].strip(), 'hook': hook, 'scenes': normalized}


def fallback_story():
    # Filmowy scenariusz „co by było, gdyby…”:
    # deadline + ograniczone zasoby + konsekwencje + decyzja.
    prompts = [
        'Abandoned city railway station at blue hour, wet pavement, one distant platform light, cinematic grounded realism.',
        'Adult survivor in a worn railway jacket holding a folded paper city map at a station entrance, medium shot, face and hands visible.',
        'Adult female survivor in a faded olive coat leaning over a station map on a desk, medium shot, face and hands visible.',
        'Old railway radio receiver on a control desk, one small indicator glowing, close-up, no readable labels.',
        'Adult male survivor moving along an empty railway platform, medium shot with upper body and face visible, cold dawn light.',
        'Adult female survivor beside an old station notice board, holding a paper passenger list, medium shot, face and hands visible.',
        'An abandoned platform disappearing into mist, a distant train headlight appearing between buildings, realistic cinematic frame.',
        'A close-up of a wristwatch beside a paper map and a nearly empty flashlight, no readable screen or writing.',
        'A deserted station control room with a route map and a single emergency light, no readable text.',
        'A long station corridor with one open door at the far end, grounded unsettling atmosphere.',
    ]
    raw_scenes = [
        ('narrator', 'calm_dark',
         'O ósmej rano całe miasto dostało ten sam komunikat: dotrzyj do punktu ewakuacji przed zmrokiem.',
         'environment', 'none', '',
         'Slow push past the empty station clock toward the deserted platform. Wind moves loose paper and a hanging timetable; keep the environment visibly alive.'),
        ('ocalaly_1', 'tense_male',
         'Adres nie istnieje. W tym miejscu od dwudziestu lat nie ma już stacji.',
         'talking_character', 'survivor_1', CHARACTER_BIBLE['survivor_1'],
         'The survivor unfolds the paper map with both hands, traces a route with one finger, then looks up sharply toward the camera as if realizing something is wrong. Keep his face, mouth, shoulders and hands visible throughout.'),
        ('ocalaly_2', 'calm_female',
         'To nie jest punkt ewakuacji. Ktoś dopisał go do mapy dopiero dzisiaj.',
         'talking_character', 'survivor_2', CHARACTER_BIBLE['survivor_2'],
         'The survivor steps close to the desk, leans over the map and points firmly to a newly marked location, then turns her head toward the other survivor. Keep her face and mouth visible while her arms and upper body move naturally.'),
        ('glos_z_radia', 'radio_distorted',
         'Jeśli jesteście na trzecim peronie, nie czekajcie na pociąg. On nie jedzie po was.',
         'environment', 'none', '',
         'The radio indicator flickers as static crackles; the camera slowly moves closer while dust vibrates on the desk. No speaking face in frame.'),
        ('ocalaly_1', 'tense_male',
         'Słyszysz? Ten pociąg nie ma maszynisty. A mimo to właśnie zahamował.',
         'talking_character', 'survivor_1', CHARACTER_BIBLE['survivor_1'],
         'The survivor takes two cautious steps toward the platform, stops abruptly at the sound of brakes, turns his head and shoulders toward the tracks, then raises one hand to signal silence. Keep his face and mouth visible; use a medium close-up that includes upper-body movement.'),
        ('ocalaly_2', 'calm_female',
         'Na liście jest nasze nazwisko. Ale przy moim ktoś dopisał: nie wpuszczać.',
         'talking_character', 'survivor_2', CHARACTER_BIBLE['survivor_2'],
         'The survivor lifts the paper list toward the camera, scans it with her eyes, lowers it slightly and slowly looks up with a worried expression. Her fingers visibly tremble and her shoulders tense; keep her face and mouth unobstructed.'),
        ('narrator', 'calm_dark',
         'Mieli osiem godzin. Zostało im jedenaście minut. A pociąg otworzył drzwi.',
         'environment', 'none', '',
         'The train doors slide open by themselves in the distance as mist rolls across the platform; a loose strip of paper skitters toward the camera. No visible speaking face.'),
    ]
    scenes = []
    for index, (speaker, role, text, shot_type, character_id, character_description, motion_prompt) in enumerate(raw_scenes[:NUM_SCENES], start=1):
        scenes.append({
            'scene_id': f'S{index:02d}',
            'speaker': speaker,
            'voice_role': role,
            'text': text,
            'visual_prompt': prompts[index-1],
            'motion_prompt': motion_prompt,
            'shot_type': shot_type,
            'character_id': character_id,
            'character_description': character_description,
        })
    hook = {
        'scene_id': 'H00',
        'speaker': 'glos_z_radia',
        'voice_role': 'radio_distorted',
        'text': 'Masz osiem godzin. Nie ufaj pociągowi.',
        'visual_prompt': prompts[0],
        'motion_prompt': 'The station clock ticks as loose paper flutters across the empty platform; a distant warning light pulses through light fog. No person speaking on camera.',
        'shot_type': 'environment',
        'character_id': 'none',
        'character_description': '',
    }
    return validate_story({'title': 'Osiem godzin do ewakuacji', 'hook': hook, 'scenes': scenes})


def generate_story_openai():
    if not OPENAI_API_KEY:
        raise RuntimeError('Brak OPENAI_API_KEY.')
    system_prompt = (
        'Jesteś scenarzystą mrocznych polskich shortów survival/apokalipsa. '
        'Zwróć wyłącznie poprawny JSON. Tytuł, hook i wszystkie wypowiadane kwestie muszą być po polsku. '
        'Dialogi są krótkie, naturalne i przeplatane oszczędną narracją. '
        'W każdym odcinku pojawia się ktoś lub coś, czego nie powinno być w danej lokacji, '
        'oraz ukryty obiekt, pomieszczenie albo zapis nieobecny w żadnym wykazie. '
        'Klimat: filmowy scenariusz co-by-było-gdyby, deadline, ograniczone zasoby, rosnące konsekwencje i trudny wybór. '
        'Bez romansów, humoru, lekkiego tonu i graficznej przemocy. '
        'Prompty visual_prompt, motion_prompt i character_description pisz po angielsku; wypowiedzi text wyłącznie po polsku. ' 
        'Każda scena ma mieć konkretną akcję: bohater chodzi, obraca się, gestykuluje, podnosi lub odkłada przedmiot albo reaguje na zdarzenie. Sam ruch kamery nie wystarcza.'
    )
    user_prompt = f"""Stwórz odcinek serii {SERIES_NAME}.
Temat: {EPISODE_TOPIC}
Lokacja: {EPISODE_LOCATION}
Styl wizualny: {VISUAL_STYLE}
Czas docelowy: około {TARGET_SECONDS} sekund.
Zwróć obiekt JSON z polskim title, obiektem hook i dokładnie {NUM_SCENES} elementami scenes.

Format każdej kwestii:
{{
  "scene_id": "H00 lub S01",
  "speaker": "narrator | ocalaly_1 | ocalaly_2 | glos_z_radia | nieznany",
  "voice_role": "calm_dark | tense_male | calm_female | radio_distorted | unknown",
  "text": "krótka naturalna wypowiedź po polsku",
  "visual_prompt": "English cinematic start-frame prompt, portrait 9:16, no text",
  "motion_prompt": "English description of one continuous visible physical action from first frame to end of clip",
  "shot_type": "talking_character albo environment",
  "character_id": "survivor_1, survivor_2, unknown_entity albo none",
  "character_description": "English stable adult character description; empty for environment"
}}

Dla widocznych wypowiedzi ocalałych/nieznanego użyj talking_character; twarz dorosłej fikcyjnej postaci ma być wyraźna, stabilna i front-facing. Dla narratora i głosu z radia użyj environment. Zachowuj ten sam character_id i opis postaci w kolejnych scenach. Hook ma mieć maksymalnie 9 słów i mocno otworzyć pierwsze 3 sekundy. Każdy text ma mieć naturalną interpunkcję. Każdy motion_prompt musi opisywać widoczny ruch postaci albo otoczenia, a nie tylko ruch kamery. Dla talking_character zachowaj twarz i usta w kadrze podczas ruchu głowy, rąk i tułowia. Nie dopisuj pól i nie zmieniaj liczby scen.
"""
    response = requests.post(
        f'{OPENAI_BASE_URL}/chat/completions',
        headers={'Authorization': f'Bearer {OPENAI_API_KEY}', 'Content-Type': 'application/json'},
        json={
            'model': OPENAI_CHAT_MODEL,
            'response_format': {'type': 'json_object'},
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            'temperature': 0.85,
        },
        timeout=120,
    )
    response.raise_for_status()
    content = response.json()['choices'][0]['message']['content']
    return validate_story(json.loads(strip_json_fence(content)))


try:
    if not TEST_MODE and LLM_PROVIDER == 'openai_compatible' and OPENAI_API_KEY:
        STORY = generate_story_openai()
        print('[OK] Dynamiczny scenariusz wygenerowany przez model.')
    else:
        STORY = fallback_story()
        print('[INFO] Użyto polskiego scenariusza demonstracyjnego.')
except Exception as exc:
    print(f'[OSTRZEŻENIE] Generator scenariusza nie powiódł się: {type(exc).__name__}: {exc}')
    print('[INFO] Przechodzę na sprawdzony scenariusz demonstracyjny.')
    STORY = fallback_story()

SEGMENT_SPECS = [STORY['hook']] + STORY['scenes']
STORY_PATH = ASSETS_DIR / f'{EPISODE_ID}_story.json'
STORY_PATH.write_text(json.dumps(STORY, ensure_ascii=False, indent=2), encoding='utf-8')

print('\nTYTUŁ:', STORY['title'])
for item in SEGMENT_SPECS:
    print(
        f"{item['scene_id']} | {item['speaker']} | {item['shot_type']} | "
        f"{item['voice_role']}: {item['text']}"
    )
print(f'[OK] Polski scenariusz JSON: {STORY_PATH}')


In [ ]:
# KOMÓRKA 4/7 — osobne pliki TTS dla każdej kwestii

import asyncio

VOICE_PROFILES = {
    'calm_dark': {'voice': 'pl-PL-MarekNeural', 'rate': '-5%', 'pitch': '-7Hz'},
    'tense_male': {'voice': 'pl-PL-MarekNeural', 'rate': '+1%', 'pitch': '-9Hz'},
    'calm_female': {'voice': 'pl-PL-ZofiaNeural', 'rate': '-3%', 'pitch': '-4Hz'},
    'radio_distorted': {'voice': 'pl-PL-AgnieszkaNeural', 'rate': '-6%', 'pitch': '-10Hz'},
    'unknown': {'voice': 'pl-PL-MarekNeural', 'rate': '-8%', 'pitch': '-12Hz'},
}

SPEAKER_LABELS_PL = {
    'narrator': 'DZIENNIK',
    'ocalaly_1': 'OCALAŁY 1',
    'ocalaly_2': 'OCALAŁA 2',
    'glos_z_radia': 'GŁOS Z RADIA',
    'nieznany': 'NIEZNANY',
}


def normalize_edge_tts_adjustment(value, unit):
    """Normalizuje rate/pitch do formatu wymaganego przez Edge TTS; wartości neutralne pomija."""
    if value is None:
        return None

    raw = str(value).strip()
    if raw.casefold() in {'', 'default', '0', '0%', '0hz'}:
        return None

    match = re.fullmatch(r'([+-]?\d+(?:[.,]\d+)?)\s*(%|hz)?', raw, flags=re.IGNORECASE)
    if not match:
        raise ValueError(
            f"Niepoprawna wartość parametru Edge TTS: {value!r}; oczekiwano liczby z jednostką {unit}."
        )

    number = float(match.group(1).replace(',', '.'))
    if number == 0:
        return None

    supplied_unit = match.group(2)
    if supplied_unit and supplied_unit.casefold() != unit.casefold():
        raise ValueError(
            f"Niepoprawna jednostka Edge TTS: {value!r}; dla tego parametru wymagana jest jednostka {unit}."
        )

    sign = '+' if number > 0 else '-'
    return f"{sign}{abs(number):g}{unit}"


# Szybki test normalizatora — bez połączenia z usługą TTS.
assert normalize_edge_tts_adjustment(None, '%') is None
assert normalize_edge_tts_adjustment('default', 'Hz') is None
assert normalize_edge_tts_adjustment('0%', '%') is None
assert normalize_edge_tts_adjustment('0Hz', 'Hz') is None
assert normalize_edge_tts_adjustment('5', '%') == '+5%'
assert normalize_edge_tts_adjustment('-5Hz', 'Hz') == '-5Hz'


def _is_retryable_edge_tts_error(exc):
    """Rozpoznaje brak audio i typowe przejściowe błędy sieciowe."""
    error_name = type(exc).__name__.casefold()
    error_text = str(exc).casefold()

    if error_name == 'noaudioreceived':
        return True
    if isinstance(exc, (asyncio.TimeoutError, TimeoutError, ConnectionError, OSError)):
        return True

    try:
        import aiohttp
        if isinstance(exc, aiohttp.ClientError):
            return True
    except Exception:
        pass

    transient_markers = (
        'no audio was received',
        'temporarily unavailable',
        'server disconnected',
        'connection reset',
        'connection aborted',
        'connection refused',
        'timed out',
        'timeout',
        'network is unreachable',
        'name or service not known',
        '429', '502', '503', '504',
    )
    return any(marker in error_text for marker in transient_markers)


async def create_edge_tts_segment(item, mp3_path):
    scene_id = str(item.get('scene_id') or '<brak scene_id>')
    voice_role = item.get('voice_role')
    profile = VOICE_PROFILES.get(voice_role)
    if not profile:
        raise ValueError(f"{scene_id}: nieznany voice_role={voice_role!r}.")

    voice = profile.get('voice')
    clean_text = str(item.get('text') or '').strip()
    if not clean_text or not any(char.isalnum() for char in clean_text):
        raise ValueError(
            f"{scene_id}: tekst dla Edge TTS jest pusty lub zawiera wyłącznie interpunkcję/znaki specjalne."
        )

    # Parametry neutralne pomijamy całkowicie — nie wysyłamy np. rate='0%' ani pitch='0Hz'.
    try:
        normalized_rate = normalize_edge_tts_adjustment(profile.get('rate'), '%')
        normalized_pitch = normalize_edge_tts_adjustment(profile.get('pitch'), 'Hz')
    except ValueError as exc:
        raise ValueError(f"{scene_id}: niepoprawne ustawienia głosu {voice!r}: {exc}") from exc

    try:
        import edge_tts
    except ImportError as exc:
        raise RuntimeError(
            "Brak biblioteki edge-tts. Uruchom komórkę instalacyjną przed komórką TTS."
        ) from exc

    tts_kwargs = {
        'text': clean_text,
        'voice': voice,
        'volume': '+0%',
    }
    if normalized_rate is not None:
        tts_kwargs['rate'] = normalized_rate
    if normalized_pitch is not None:
        tts_kwargs['pitch'] = normalized_pitch

    mp3_path = Path(mp3_path)
    max_attempts = 3
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            # Usuwamy ewentualny plik częściowy z poprzedniej próby.
            if mp3_path.exists():
                mp3_path.unlink()
            mp3_path.parent.mkdir(parents=True, exist_ok=True)

            communicate = edge_tts.Communicate(**tts_kwargs)
            await communicate.save(str(mp3_path))

            if not mp3_path.exists() or mp3_path.stat().st_size <= 0:
                raise RuntimeError('Edge TTS nie zapisał audio lub zwrócił pusty plik MP3.')

            return mp3_path

        except Exception as exc:
            last_error = exc
            try:
                if mp3_path.exists():
                    mp3_path.unlink()
            except OSError:
                pass

            if attempt >= max_attempts or not _is_retryable_edge_tts_error(exc):
                break

            delay_seconds = min(1.0 * attempt, 3.0)
            print(
                f"[TTS RETRY] {scene_id}, próba {attempt}/{max_attempts}, "
                f"{type(exc).__name__}; ponawiam za {delay_seconds:.1f}s."
            )
            await asyncio.sleep(delay_seconds)

    # Podgląd tekstu nie zawiera żadnych sekretów/parametrów połączenia.
    preview = re.sub(r'\s+', ' ', clean_text[:100]).strip()
    if len(clean_text) > 100:
        preview += '…'
    raise RuntimeError(
        "Edge TTS nie wygenerował poprawnego MP3 po "
        f"{attempt} próbach | scene_id={scene_id!r} | voice={voice!r} | "
        f"rate={normalized_rate!r} | pitch={normalized_pitch!r} | "
        f"długość_tekstu={len(clean_text)} | tekst_początek={preview!r} | "
        f"błąd={type(last_error).__name__ if last_error else 'UnknownError'}."
    ) from last_error

def convert_tts_to_wav(mp3_path, wav_path, item):
    is_radio = (
        item['voice_role'] in {'radio_distorted', 'unknown'}
        or item['speaker'] in {'glos_z_radia', 'nieznany'}
    )
    if is_radio:
        filters = (
            '[0:a]aresample=48000,highpass=f=240,lowpass=f=3600,'
            'acompressor=threshold=0.09:ratio=2.4:attack=12:release=130[voice];'
            '[1:a]highpass=f=850,lowpass=f=4200,'
            f'volume={RADIO_STATIC_GAIN:.3f}[static];'
            '[voice][static]amix=inputs=2:duration=first:normalize=0,alimiter=limit=0.95[out]'
        )
        run_cmd([
            'ffmpeg', '-y', '-i', str(mp3_path),
            '-f', 'lavfi', '-i', 'anoisesrc=color=pink:amplitude=0.008:sample_rate=48000',
            '-filter_complex', filters, '-map', '[out]',
            '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(wav_path)
        ])
    else:
        run_cmd([
            'ffmpeg', '-y', '-i', str(mp3_path),
            '-af', 'aresample=48000,highpass=f=70,lowpass=f=16000,'
                   'acompressor=threshold=0.10:ratio=2.0:attack=18:release=220,'
                   'alimiter=limit=0.95',
            '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(wav_path)
        ])
    if not wav_path.exists() or wav_path.stat().st_size < 4000:
        raise RuntimeError(f'Nie powstał poprawny plik audio {wav_path.name}.')


if TTS_PROVIDER != 'edge':
    raise NotImplementedError(
        'TTS_PROVIDER=external jest miejscem na adapter zewnętrzny; domyślny, gotowy adapter to edge-tts.'
    )

# Ustaw na "H00", aby przetestować wyłącznie problematyczny segment.
# Przed pełnym renderem przywróć None.
TTS_TEST_SCENE_ID = None
TTS_ITEMS_TO_PROCESS = [
    item for item in SEGMENT_SPECS
    if TTS_TEST_SCENE_ID is None or item['scene_id'] == TTS_TEST_SCENE_ID
]
if TTS_TEST_SCENE_ID is not None and not TTS_ITEMS_TO_PROCESS:
    raise ValueError(f"Nie znaleziono segmentu TTS o scene_id={TTS_TEST_SCENE_ID!r}.")

TIMELINE = []
cursor = 0.0
for index, item in enumerate(TTS_ITEMS_TO_PROCESS):
    mp3_path = AUDIO_DIR / f"{EPISODE_ID}_{index:02d}_{item['scene_id']}.mp3"
    wav_path = AUDIO_DIR / f"{EPISODE_ID}_{index:02d}_{item['scene_id']}.wav"
    await create_edge_tts_segment(item, mp3_path)
    if not mp3_path.exists() or mp3_path.stat().st_size < 1000:
        raise RuntimeError(f'TTS zwrócił pusty plik: {mp3_path.name}')
    convert_tts_to_wav(mp3_path, wav_path, item)
    duration = probe_duration(wav_path)
    pause = PAUSE_BETWEEN_LINES if index < len(TTS_ITEMS_TO_PROCESS) - 1 else 0.0
    TIMELINE.append({
        'scene_id': item['scene_id'],
        'audio_path': str(wav_path),
        'duration': duration,
        'start': cursor,
        'pause_after': pause,
        'speaker': item['speaker'],
        'text': item['text'],
    })
    cursor += duration + pause
    print(
        f"[OK] {item['scene_id']} | {SPEAKER_LABELS_PL[item['speaker']]} | "
        f"{VOICE_PROFILES[item['voice_role']]['voice']} | {duration:.2f}s"
    )

TOTAL_DIALOGUE_SECONDS = cursor
print(f'[OK] Oddzielne pliki audio: {len(TIMELINE)}')
print(f'[INFO] Czas dialogów z pauzami: {TOTAL_DIALOGUE_SECONDS:.2f}s')


In [ ]:
# KOMÓRKA 5/7 — FLUX: realistyczne obrazy startowe (pomijane w TEST_MODE)

from io import BytesIO
from PIL import Image


def normalize_portrait_image(source_path, output_path):
    with Image.open(source_path) as original:
        image = original.convert('RGB')
    target_ratio = VIDEO_WIDTH / VIDEO_HEIGHT
    width, height = image.size
    ratio = width / height
    if ratio > target_ratio:
        crop_w = int(height * target_ratio)
        left = (width - crop_w) // 2
        image = image.crop((left, 0, left + crop_w, height))
    else:
        crop_h = int(width / target_ratio)
        top = (height - crop_h) // 2
        image = image.crop((0, top, width, top + crop_h))
    image = image.resize((VIDEO_WIDTH, VIDEO_HEIGHT), Image.Resampling.LANCZOS)
    image.save(output_path, format='PNG', optimize=True)
    return Path(output_path)


FRAME_RECORDS = []
if TEST_MODE:
    print('[TEST_MODE] POMINIĘTO FLUX — nie wygenerowano obrazów i nie wysłano płatnych zadań.')
    print('[TEST_MODE] W trybie pełnym każda kwestia otrzyma realistyczny kadr startowy z FLUX.1 Dev.')
else:
    if not FAL_KEY:
        raise RuntimeError('Pełne generowanie obrazów wymaga FAL_KEY.')
    for index, item in enumerate(SEGMENT_SPECS, start=0):
        character = item['character_description']
        prompt = (
            f"{item['visual_prompt']} {character}. Action staging for the opening frame: {item['motion_prompt']}. "
            f"Visual style: {VISUAL_STYLE}. "
            "Photorealistic cinematic still, vertical portrait composition, 9:16. "
            "No text, no letters, no logos, no subtitles, no watermark. "
            "Survival horror atmosphere, fictional adult characters only, non-graphic."
        )
        last_error = None
        for attempt in range(1, IMAGE_RETRIES + 1):
            try:
                image_result, request_id = fal_queue_call(
                    FAL_IMAGE_MODEL,
                    {
                        'prompt': prompt,
                        'image_size': {'width': 768, 'height': 1365},
                        'num_images': 1,
                        'output_format': 'png',
                        'enable_safety_checker': True,
                    },
                    task_name=f'FLUX {item["scene_id"]}',
                    max_wait_seconds=FAL_MAX_WAIT_SECONDS,
                )
                images = image_result.get('images') or []
                if not images or not images[0].get('url'):
                    raise ValueError(f'FLUX nie zwrócił images[0].url: {image_result}')
                remote_url = images[0]['url']
                raw_path = FRAME_EPISODE_DIR / f"{index:02d}_{item['scene_id']}_raw.png"
                download_remote_file(remote_url, raw_path)
                frame_path = FRAME_EPISODE_DIR / f"{index:02d}_{item['scene_id']}.png"
                normalize_portrait_image(raw_path, frame_path)
                FRAME_RECORDS.append({
                    'scene_id': item['scene_id'],
                    'image_path': str(frame_path),
                    'image_url': remote_url,
                    'request_id': request_id,
                })
                print(f'[OK] FLUX {item["scene_id"]}: {frame_path.name}')
                break
            except Exception as exc:
                last_error = exc
                print(f'[OSTRZEŻENIE] FLUX {item["scene_id"]} próba {attempt}/{IMAGE_RETRIES}: {exc}')
                if attempt < IMAGE_RETRIES:
                    time.sleep(min(2 ** attempt, 10))
        else:
            raise RuntimeError(f'Nie wygenerowano obrazu {item["scene_id"]}: {last_error}')

    if len(FRAME_RECORDS) != len(SEGMENT_SPECS):
        raise RuntimeError('Nie wszystkie kwestie otrzymały obraz startowy.')

FRAME_MANIFEST = ASSETS_DIR / f'{EPISODE_ID}_frames.json'
FRAME_MANIFEST.write_text(json.dumps(FRAME_RECORDS, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
# KOMÓRKA 6/7 — Kling V3 MULTI-SHOT + Sync Lipsync 2

def make_shot_durations(total_seconds, target_seconds=SHOT_DURATION_TARGET, max_shots=MAX_SHOTS_PER_SEGMENT):
    """Dzieli cały klip na krótkie ujęcia; suma trwa tyle sekund co planowany klip."""
    total = int(total_seconds)
    if total < 3 or total > 15:
        raise ValueError(f'Kling multi-shot: całkowity czas musi wynosić 3–15 s, otrzymano {total}.')
    shot_count = min(max_shots, max(2, int(math.ceil(total / target_seconds))))
    shot_count = min(shot_count, total)
    base, remainder = divmod(total, shot_count)
    durations = [base + (1 if index < remainder else 0) for index in range(shot_count)]
    if sum(durations) != total or min(durations) < 1:
        raise RuntimeError(f'Niepoprawny plan ujęć: {durations}.')
    return durations


def build_kling_multi_prompt(item, shot_durations):
    """Tworzy osobne prompty montażowe dla kolejnych cięć w Kling V3."""
    is_character = item['shot_type'] == 'talking_character'
    base_action = item['motion_prompt']
    continuity = (
        "Same fictional adult character, same clothes and location. "
        if is_character else
        "Same location, time of day and visual atmosphere across all shots. "
    )
    if is_character:
        shot_styles = [
            "MEDIUM ACTION SHOT: keep the speaker's face and mouth visible while the character performs the described physical action with clear shoulder, arm and hand movement.",
            "INSERT CUTAWAY: cut close to the character's hands and the relevant prop or surface as the action continues; show purposeful finger and hand movement.",
            "SIDE FOLLOW SHOT: show the same character turning head and shoulders, shifting position or taking a step; the actor's body movement must be obvious.",
            "REACTION CLOSE SHOT: return to the character's face and show a clear emotional reaction with natural mouth movement; keep face and mouth unobstructed.",
            "OVER-THE-SHOULDER ACTION SHOT: show the character reaching, pointing or moving an object with a visible change in posture.",
            "WIDER FOLLOW-THROUGH SHOT: show the character completing the action with a step, turn or gesture; actor movement, not camera movement, leads the shot."
        ]
    else:
        shot_styles = [
            "WIDE ESTABLISHING SHOT: reveal the full location with visible environmental movement, not a still postcard.",
            "DETAIL INSERT: cut close to a meaningful object, paper, radio, clock, light or surface; show it moving, blinking or reacting.",
            "LATERAL TRACKING SHOT: travel along the location and reveal depth, moving fog, rain, dust or practical lights.",
            "SUSPENSE REVEAL SHOT: reveal a new area, opening door, approaching light or moving silhouette through the environment.",
            "LOW ANGLE DETAIL SHOT: show a moving foreground element and strong depth into the location.",
            "WIDE FOLLOW-THROUGH SHOT: show the environment changing in a visible way while preserving the same place and time."
        ]

    result = []
    count = len(shot_durations)
    for index, duration in enumerate(shot_durations):
        style = shot_styles[index % len(shot_styles)]
        prompt = (
            f"SHOT {index + 1} OF {count}, duration {duration} seconds. "
            f"{continuity}{style} "
            f"Physical action / motion direction: {base_action} "
            f"Visual reference: {item['visual_prompt']} "
            f"Character reference: {item.get('character_description', '')}. "
            "Cinematic live-action realism, purposeful movement, coherent anatomy and physics, "
            "clean edit point at the end of this shot, no captions, no logos, no watermark."
        )
        result.append({'prompt': prompt, 'duration': str(duration)})
    if len({shot['prompt'] for shot in result}) != len(result):
        raise RuntimeError(f"{item['scene_id']}: prompty ujęć nie są unikalne.")
    return result


VIDEO_RECORDS = []

if TEST_MODE:
    print('[TEST_MODE] POMINIĘTO Kling Image-to-Video i Sync Lipsync 2.')
    print('[TEST_MODE] Nie powstaną animowane klipy ani klipy z lip-sync. Nie jest generowany zastępczy film.')
else:
    if not FAL_KEY:
        raise RuntimeError('Pełne generowanie wideo wymaga FAL_KEY.')
    if len(FRAME_RECORDS) != len(SEGMENT_SPECS):
        raise RuntimeError('Brak obrazów startowych — uruchom komórkę FLUX przed generowaniem wideo.')

    for item, frame_record, timeline in zip(SEGMENT_SPECS, FRAME_RECORDS, TIMELINE):
        audio_duration = float(timeline['duration'])
        requested_total_seconds = int(math.ceil(audio_duration + timeline['pause_after']))
        if requested_total_seconds > 15:
            raise ValueError(
                f"{item['scene_id']}: kwestia trwa {audio_duration:.1f}s. "
                "Kling V3 multi-shot obsługuje maksymalnie 15 sekund na klip — skróć kwestię."
            )
        requested_total_seconds = max(3, requested_total_seconds)
        shot_durations = make_shot_durations(requested_total_seconds)
        multi_prompt = build_kling_multi_prompt(item, shot_durations)
        requested_duration = str(requested_total_seconds)
        print(
            f"[VIDEO] {item['scene_id']} | {item['shot_type']} | "
            f"{len(shot_durations)} ujęcia: {shot_durations} s | razem {requested_duration}s"
        )

        image_url = fal_upload_file(frame_record['image_path'])

        video_result = None
        last_error = None
        for attempt in range(1, VIDEO_RETRIES + 1):
            try:
                video_result, i2v_request_id = fal_queue_call(
                    FAL_I2V_MODEL,
                    {
                        'start_image_url': image_url,
                        'multi_prompt': multi_prompt,
                        'shot_type': 'customize',
                        'duration': requested_duration,
                        'generate_audio': False,
                        'enable_safety_checker': True,
                        'negative_prompt': 'static person, frozen pose, still frame, no body movement, camera movement only, low quality, unstable face, identity drift, scene cut, text, logo, watermark, graphic gore',
                    },
                    task_name=f'Kling I2V {item["scene_id"]}',
                    max_wait_seconds=FAL_MAX_WAIT_SECONDS,
                )
                break
            except Exception as exc:
                last_error = exc
                print(f'[OSTRZEŻENIE] Kling {item["scene_id"]} próba {attempt}/{VIDEO_RETRIES}: {exc}')
                if attempt < VIDEO_RETRIES:
                    time.sleep(min(2 ** attempt, 12))
        if video_result is None:
            raise RuntimeError(f'Kling I2V nie wygenerował {item["scene_id"]}: {last_error}')

        raw_video_url = (video_result.get('video') or {}).get('url')
        if not raw_video_url:
            raise ValueError(f'Kling nie zwrócił video.url dla {item["scene_id"]}: {video_result}')
        raw_clip_path = VIDEO_EPISODE_DIR / f"{item['scene_id']}_i2v.mp4"
        download_remote_file(raw_video_url, raw_clip_path)

        final_video_url = raw_video_url
        lip_request_id = None
        used_lipsync = False

        if item['shot_type'] == 'talking_character':
            audio_url = fal_upload_file(timeline['audio_path'])
            final_video_url = None
            lip_request_id = None
            lip_result = None
            last_lip_error = None
            for lip_attempt in range(1, VIDEO_RETRIES + 1):
                try:
                    lip_result, lip_request_id = fal_queue_call(
                        FAL_LIPSYNC_MODEL,
                        {
                            'model': LIPSYNC_MODEL_VARIANT,
                            'video_url': raw_video_url,
                            'audio_url': audio_url,
                            'sync_mode': 'cut_off',
                            'enable_safety_checker': True,
                        },
                        task_name=f'Sync Lipsync {item["scene_id"]} (próba {lip_attempt})',
                        max_wait_seconds=FAL_MAX_WAIT_SECONDS,
                    )
                    final_video_url = (lip_result.get('video') or {}).get('url')
                    if not final_video_url:
                        raise ValueError(f'Sync Lipsync nie zwrócił video.url: {lip_result}')
                    break
                except Exception as exc:
                    last_lip_error = exc
                    print(f'[OSTRZEŻENIE] Lip-sync {item["scene_id"]} próba {lip_attempt}/{VIDEO_RETRIES}: {exc}')
                    if lip_attempt < VIDEO_RETRIES:
                        time.sleep(min(2 ** lip_attempt, 12))
            if not final_video_url:
                raise RuntimeError(f'Lip-sync nie powiódł się po {VIDEO_RETRIES} próbach: {last_lip_error}')
            used_lipsync = True
            clip_path = VIDEO_EPISODE_DIR / f"{item['scene_id']}_lipsync.mp4"
            download_remote_file(final_video_url, clip_path)
            print(f'[OK] {item["scene_id"]}: lip-sync zakończony; {len(shot_durations)} krótkie ujęcia.')
        else:
            clip_path = VIDEO_EPISODE_DIR / f"{item['scene_id']}_environment.mp4"
            shutil.copy2(raw_clip_path, clip_path)
            print(f'[OK] {item["scene_id"]}: {len(shot_durations)} ujęcia otoczenia, bez lip-sync.')

        clip_probe = run_cmd([
            'ffprobe', '-v', 'error', '-show_entries', 'stream=codec_type,width,height',
            '-of', 'json', str(clip_path)
        ])
        clip_data = json.loads(clip_probe.stdout)
        if not any(stream.get('codec_type') == 'video' for stream in clip_data.get('streams', [])):
            raise RuntimeError(f'Pobrany klip {item["scene_id"]} nie zawiera wideo.')

        VIDEO_RECORDS.append({
            'scene_id': item['scene_id'],
            'clip_path': str(clip_path),
            'shot_type': item['shot_type'],
            'lipsync': used_lipsync,
            'i2v_request_id': i2v_request_id,
            'lipsync_request_id': lip_request_id,
            'duration': audio_duration,
            'shot_count': len(shot_durations),
            'shot_durations': shot_durations,
        })

    if len(VIDEO_RECORDS) != len(SEGMENT_SPECS):
        raise RuntimeError('Nie wszystkie sceny otrzymały klip wideo.')

VIDEO_MANIFEST = ASSETS_DIR / f'{EPISODE_ID}_video_manifest.json'
VIDEO_MANIFEST.write_text(json.dumps(VIDEO_RECORDS, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
# KOMÓRKA 7/7 — FFmpeg, napisy PL, ambient i końcowa walidacja ffprobe

from IPython.display import display, Video


def srt_timestamp(seconds):
    milliseconds = max(0, int(round(seconds * 1000)))
    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, milliseconds = divmod(remainder, 1000)
    return f'{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}'


def wrap_caption(text, width=30):
    return '\n'.join(textwrap.wrap(
        text.strip(), width=width,
        break_long_words=False, break_on_hyphens=False
    ))


if TEST_MODE:
    test_report = {
        'episode_id': EPISODE_ID,
        'test_mode': True,
        'mp4_created': False,
        'title': STORY['title'],
        'target_shot_duration_seconds': SHOT_DURATION_TARGET,
        'segments': [
            {
                'scene_id': item['scene_id'],
                'speaker': item['speaker'],
                'shot_type': item['shot_type'],
                'text': item['text'],
                'audio_file': timeline['audio_path'],
                'planned_shot_durations': make_shot_durations(
                    max(3, int(math.ceil(float(timeline['duration']) + float(timeline['pause_after']))))
                ),
                'skipped': ['FLUX image generation', 'Kling image-to-video'] + (
                    ['Sync Lipsync 2'] if item['shot_type'] == 'talking_character' else []
                ),
            }
            for item, timeline in zip(SEGMENT_SPECS, TIMELINE)
        ],
        'planned_total_shots': sum(
            len(make_shot_durations(
                max(3, int(math.ceil(float(timeline['duration']) + float(timeline['pause_after']))))
            ))
            for timeline in TIMELINE
        ),
        'reason': 'TEST_MODE=True: no paid video jobs were submitted; no MP4 was rendered.',
    }
    report_path = ASSETS_DIR / f'{EPISODE_ID}_test_report.json'
    report_path.write_text(json.dumps(test_report, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\n[TEST_MODE] Plan testowy i pliki audio zostały zapisane.')
    print('[POMINIĘTO] Nie utworzono animowanych klipów, lip-sync ani MP4.')
    print('[INFO] Aby włączyć pełne generowanie, ustaw TEST_MODE=False i dodaj FAL_KEY.')
    print('[INFO] Raport:', report_path)

else:
    if len(VIDEO_RECORDS) != len(SEGMENT_SPECS):
        raise RuntimeError('Brakuje animowanych klipów dla części scen — przerwano eksport.')
    if len(TIMELINE) != len(SEGMENT_SPECS):
        raise RuntimeError('Brakuje audio dla części scen — przerwano eksport.')

    segment_video_paths = []
    subtitle_entries = []

    for index, (item, timeline, video_record) in enumerate(zip(SEGMENT_SPECS, TIMELINE, VIDEO_RECORDS)):
        audio_path = Path(timeline['audio_path'])
        clip_path = Path(video_record['clip_path'])
        if not audio_path.exists() or audio_path.stat().st_size < 4000:
            raise RuntimeError(f'Brak poprawnego audio: {audio_path}')
        if not clip_path.exists() or clip_path.stat().st_size < 100_000:
            raise RuntimeError(f'Brak poprawnego klipu: {clip_path}')

        speech_duration = float(timeline['duration'])
        pause_after = float(timeline['pause_after'])
        segment_total = speech_duration + pause_after
        segment_path = TEMP_DIR / f'{EPISODE_ID}_segment_{index:02d}.mp4'

        video_filter = (
            f'scale={VIDEO_WIDTH}:{VIDEO_HEIGHT}:force_original_aspect_ratio=increase,'
            f'crop={VIDEO_WIDTH}:{VIDEO_HEIGHT},fps={FPS},setsar=1,'
            f'tpad=stop_mode=clone:stop_duration={pause_after:.3f},format=yuv420p'
        )
        audio_filter = (
            f'aresample=48000,apad=pad_dur={pause_after:.3f},'
            f'atrim=duration={segment_total:.3f},alimiter=limit=0.95'
        )
        run_cmd([
            'ffmpeg', '-y', '-stream_loop', '-1',
            '-i', str(clip_path),
            '-i', str(audio_path),
            '-vf', video_filter,
            '-af', audio_filter,
            '-map', '0:v:0', '-map', '1:a:0',
            '-t', f'{segment_total:.3f}',
            '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20',
            '-pix_fmt', 'yuv420p', '-r', str(FPS),
            '-c:a', 'aac', '-b:a', '192k', '-ar', '48000', '-ac', '2',
            str(segment_path)
        ])
        segment_video_paths.append(segment_path)

        speaker_label = SPEAKER_LABELS_PL.get(item['speaker'], 'GŁOS')
        caption = item['text'] if item['speaker'] == 'narrator' else f'{speaker_label}: {item["text"]}'
        subtitle_entries.append((
            float(timeline['start']),
            float(timeline['start']) + speech_duration,
            caption,
        ))
        print(f'[OK] Złożono segment {item["scene_id"]}: {segment_total:.2f}s')

    concat_list = TEMP_DIR / f'{EPISODE_ID}_segments.txt'
    concat_list.write_text(
        '\n'.join(f"file '{path.as_posix()}'" for path in segment_video_paths),
        encoding='utf-8'
    )
    joined_video = TEMP_DIR / f'{EPISODE_ID}_joined.mp4'
    run_cmd([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(concat_list),
        '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '19',
        '-pix_fmt', 'yuv420p', '-r', str(FPS),
        '-c:a', 'aac', '-b:a', '192k', '-ar', '48000', '-ac', '2',
        str(joined_video)
    ])

    srt_path = TEMP_DIR / f'{EPISODE_ID}_pl.srt'
    srt_lines = []
    for number, (start, end, caption) in enumerate(subtitle_entries, start=1):
        srt_lines.extend([
            str(number),
            f'{srt_timestamp(start)} --> {srt_timestamp(end)}',
            wrap_caption(caption, width=30),
            '',
        ])
    srt_path.write_text('\n'.join(srt_lines), encoding='utf-8')

    safe_srt_path = (
        srt_path.as_posix()
        .replace('\\', '\\\\')
        .replace(':', '\\:')
        .replace("'", "\\'")
    )
    subtitle_filter = (
        f"subtitles='{safe_srt_path}':"
        "force_style='FontName=DejaVu Sans,FontSize=44,PrimaryColour=&H00FFFFFF,"
        "OutlineColour=&HCC000000,BorderStyle=1,Outline=3,Shadow=1,Alignment=2,"
        "MarginV=340,MarginL=90,MarginR=90,WrapStyle=2'"
    )

    ambient_source = (
        f"anoisesrc=color=pink:amplitude=0.008:sample_rate=48000:"
        f"duration={TOTAL_DIALOGUE_SECONDS + 1.0:.3f}"
    )
    audio_mix = (
        '[0:a]aresample=48000,highpass=f=55,lowpass=f=16000,'
        'acompressor=threshold=0.10:ratio=2.0:attack=18:release=220[voice];'
        '[1:a]highpass=f=45,lowpass=f=620,'
        f'volume={AMBIENT_MIX_GAIN:.3f}[ambient];'
        "[voice][ambient]amix=inputs=2:duration=first:weights='1 0.18':normalize=0,"
        'alimiter=limit=0.95[aout]'
    )

    OUTPUT_VIDEO = OUTPUT_DIR / f'{EPISODE_ID}.mp4'
    run_cmd([
        'ffmpeg', '-y',
        '-i', str(joined_video),
        '-f', 'lavfi', '-i', ambient_source,
        '-vf', subtitle_filter,
        '-filter_complex', audio_mix,
        '-map', '0:v:0', '-map', '[aout]',
        '-c:v', 'libx264', '-preset', 'medium', '-crf', '20',
        '-pix_fmt', 'yuv420p', '-r', str(FPS),
        '-c:a', 'aac', '-b:a', '192k', '-ar', '48000', '-ac', '2',
        '-movflags', '+faststart',
        str(OUTPUT_VIDEO)
    ])

    # ffprobe: plik, rozmiar, strumienie, kodeki, rozdzielczość i długość.
    if not OUTPUT_VIDEO.exists() or OUTPUT_VIDEO.stat().st_size <= 100_000:
        raise RuntimeError(f'Plik MP4 nie istnieje albo ma nieprawidłowy rozmiar: {OUTPUT_VIDEO}')
    probe = run_cmd([
        'ffprobe', '-v', 'error',
        '-show_streams', '-show_format',
        '-of', 'json', str(OUTPUT_VIDEO)
    ])
    probe_data = json.loads(probe.stdout)
    video_stream = next(
        (stream for stream in probe_data.get('streams', []) if stream.get('codec_type') == 'video'),
        None
    )
    audio_stream = next(
        (stream for stream in probe_data.get('streams', []) if stream.get('codec_type') == 'audio'),
        None
    )
    if not video_stream:
        raise RuntimeError('Walidacja ffprobe: brak strumienia wideo.')
    if not audio_stream:
        raise RuntimeError('Walidacja ffprobe: brak strumienia audio.')
    if int(video_stream.get('width', 0)) != VIDEO_WIDTH or int(video_stream.get('height', 0)) != VIDEO_HEIGHT:
        raise RuntimeError('Walidacja ffprobe: oczekiwano obrazu 1080×1920.')
    if video_stream.get('codec_name') != 'h264':
        raise RuntimeError('Walidacja ffprobe: oczekiwano kodeka H.264.')
    if video_stream.get('r_frame_rate') != f'{FPS}/1':
        raise RuntimeError(f"Walidacja ffprobe: oczekiwano {FPS} fps, wykryto {video_stream.get('r_frame_rate')}.")
    if audio_stream.get('codec_name') != 'aac':
        raise RuntimeError('Walidacja ffprobe: oczekiwano kodeka audio AAC.')
    video_duration = float(probe_data.get('format', {}).get('duration', 0))
    try:
        audio_duration = float(audio_stream.get('duration') or video_duration)
    except (TypeError, ValueError):
        audio_duration = video_duration
    if video_duration <= 0 or audio_duration <= 0:
        raise RuntimeError('Walidacja ffprobe: niepoprawny czas trwania audio/wideo.')

    # Sprawdź, czy strumień audio zawiera sygnał, a nie wyłącznie cyfrową ciszę.
    loudness_check = run_cmd([
        'ffmpeg', '-i', str(OUTPUT_VIDEO),
        '-af', 'volumedetect', '-f', 'null', '-'
    ], check=False)
    loudness_text = (loudness_check.stderr or '') + (loudness_check.stdout or '')
    mean_match = re.search(r'mean_volume:\s*(-?inf|-?[0-9.]+)\s*dB', loudness_text)
    if not mean_match or mean_match.group(1) == '-inf':
        raise RuntimeError('Walidacja audio: strumień istnieje, ale wykryto cyfrową ciszę albo brak pomiaru.')
    print(f"[OK] Aktywny audio stream — mean_volume {mean_match.group(1)} dB")

    print('\n' + '=' * 62)
    print('[GOTOWE] Lost-words / Zgubione słowa')
    print('Tytuł:', STORY['title'])
    print('Plik:', OUTPUT_VIDEO)
    print(f'Parametry: {VIDEO_WIDTH}×{VIDEO_HEIGHT}, {FPS} fps, H.264/AAC, {video_duration:.2f}s')
    print('Lip-sync:', sum(1 for record in VIDEO_RECORDS if record['lipsync']), 'scen')
    print('Klipy otoczenia:', sum(1 for record in VIDEO_RECORDS if not record['lipsync']))
    print('Łączna liczba krótkich ujęć:', sum(record.get('shot_count', 1) for record in VIDEO_RECORDS))
    print('Napisy: DejaVu Sans | Ambient: mix z dialogiem')
    print('=' * 62)
    display(Video(str(OUTPUT_VIDEO), embed=True, width=360))
